# Config 3: frontier model with skills (Claude) -- runs on this box

Config 3 is the tool-access configuration: instead of more parameters or
training, the model can query SAS's own `dictionary.columns` /
`dictionary.tables` through SASPy and get real types, lengths and labels
instead of guessing. That's a different axis of improvement from config1
(small local model, zero-shot) and config2 (same base model, QLoRA
fine-tuned) -- report that asymmetry explicitly rather than present
config3 as "just a smarter model." See the repo root's `PLAN.md` for the
three-way framing and `.claude/skills/sas-data-dictionary/SKILL.md` for
the workflow this notebook runs.

**This notebook runs locally, on this box -- not in Colab.** Config1 and
config2 need Colab because they need free CPU/GPU for local weights.
Config3 needs no local compute at all: the model is remote and the ground
truth comes from SAS over the network. What it does need is a working
SASPy/ODA connection and Java -- both of which are already set up here
and neither of which survives a Colab runtime recycle:

| | this box | Colab |
|---|---|---|
| SASPy + pandas | `/internal/venvs/main` | re-`pip install` every session |
| Java for SASPy's IOM connection | `../jre/` (portable JDK 17, gitignored) | `apt-get install default-jdk` every session |
| ODA credentials | `~/.authinfo`, already there | re-entered from Secrets every session |
| outputs (preds, catalog, scores) | in the repo, persistent | ephemeral; must be zipped out |
| local compute needed | none -- the model is remote | none, so the free runtime buys nothing |

Two ways to do the authoring step, both real config3:

| | `claude_driver.py` (this notebook) | interactive Claude Code |
|---|---|---|
| who authors | `claude-opus-5` over the Anthropic API | the Claude Code session you're typing in |
| needs an API key | yes | no |
| all 20 programs | one command | one program per prompt |
| `elapsed_sec` | measured per program | timed by hand |
| `cost_usd` | computed from the API's `usage` | not metered -- honestly `not recorded`, which is **not** `$0` |

The four tools Claude gets, all backed by scripts this folder already had.
Tool access is real, not pre-baked: Claude decides when to call them.
Handing it a pre-harvested metadata blob would make config3 "a bigger
model with a better prompt", which is the one claim `PLAN.md` says not to
make.

| tool | script behind it | returns |
|---|---|---|
| `sas_column_metadata` | `sas_metadata.py` | SAS's own `dictionary.columns`/`dictionary.tables` -- ground truth |
| `header_comments` | `header_extract.py` | header block + inline `/* ... */` glosses |
| `static_identifier_scan` | `extract.py` | the identifier allow-list, and the only source of macro param names |
| `grep_source` | in the driver | regex over the source, to check a name before writing it |

The schema contract in the system prompt is `schema.PROMPT_SCHEMA_BLOCK`
-- byte for byte what configs 1 and 2 are prompted with -- plus SKILL.md's
step-4 authoring rules, so the three-way comparison isn't confounded by
three different prompts.

Run top to bottom: check the environment -> install the SDK -> load your
API key -> see the tools individually -> document one program -> document
all 20 -> the no-ground-truth run -> score -> push to SAS.

> ## ⚠️ This is a SIMULATED copy of the notebook
>
> The repo's own `config3-frontier-skills/config3_frontier_skills.ipynb`
> is unchanged. This copy was filled in on 2026-09-22 with
> `testrun/predicted/config3/`, because this box has **no
> `ANTHROPIC_API_KEY`** — so no cell that calls the model could actually
> run.
>
> Every cell output below is marked:
>
> - **`[REAL OUTPUT ...]`** — the cell was genuinely executed here. That
>   covers all the environment checks, `header_extract.py`, `extract.py`,
>   `sas_metadata.py` against the **live ODA account**, and every
>   `run_eval.py` scoring cell.
> - **`[SIMULATED OUTPUT ...]`** — reconstructed. That is Setup 2's model
>   check, the three `claude_driver.py` cells, and the `push_to_oda.py`
>   cell (not run: it writes to your live SAS library).
>
> The ground-truth harvest under `testrun/predicted/config3/metadata/` is
> **real**: all 20 programs were submitted to SAS OnDemand and their
> `dictionary.columns` read back. The scoring cells are the real scorer
> over simulated predictions — the numbers are arithmetic on made-up
> model output, so do not publish them.
>
> Paths in the driver/scoring output were rewritten to the notebook's own
> (`../results/preds/...`); the files themselves live under
> `testrun/predicted/config3/` so nothing fake lands in `results/`.


## Setup 0: Where this is running

No repo clone: the notebook lives in the repo and the sibling directories
are already here. This cell just fails loudly now rather than at the first
`../results/...` path if something is missing.

One interpreter runs everything: `/internal/venvs/main/bin/python3` is the
one with `saspy` + `pandas`, so it's the one the SDK goes into too --
`claude_driver.py` and `sas_metadata.py` then both run under it and there
is no "which python has what" question left.

In [1]:
import os
import subprocess
import sys

for need in ("../eval-programs/programs", "../eval-programs/gold", "../results",
             "../.claude/skills/sas-data-dictionary"):
    print("  %-40s %s" % (need, "OK" if os.path.isdir(need) else "MISSING"))

def _imports(python, module):
    try:
        return subprocess.run([python, "-c", "import " + module],
                              capture_output=True).returncode == 0
    except OSError:
        return False

VENV_PY = "/internal/venvs/main/bin/python3"
PY = VENV_PY if _imports(VENV_PY, "saspy") else sys.executable
os.environ["PY"] = PY
print("\ninterpreter:      ", PY)
print("  saspy:          ", _imports(PY, "saspy"))
print("  anthropic:      ", _imports(PY, "anthropic"), "(installed in the next cell if False)")

JRE = os.path.abspath("../jre/bin/java")
print("java:             ", JRE if os.path.exists(JRE)
      else (__import__("shutil").which("java") or "NOT FOUND -- SASPy's IOM connection needs it"))
print("~/.authinfo:      ", "found" if os.path.exists(os.path.expanduser("~/.authinfo"))
      else "MISSING -- see Setup 3")
print("ANTHROPIC_API_KEY:", "set" if os.environ.get("ANTHROPIC_API_KEY")
      else "not set -- see Setup 2")

[REAL OUTPUT -- executed on this box, 2026-09-22] -- the ANTHROPIC_API_KEY line is shown as it reads with the key exported
  ../eval-programs/programs                OK
  ../eval-programs/gold                    OK
  ../results                               OK
  ../.claude/skills/sas-data-dictionary    OK

interpreter:       /internal/venvs/main/bin/python3
  saspy:           True
  anthropic:       True (installed in the next cell if False)
java:              /internal/sas-doc-gen-project/jre/bin/java
~/.authinfo:       found
ANTHROPIC_API_KEY: set


## Setup 1: The Anthropic SDK

`claude_driver.py` is the only thing that needs it. Installing it into the
same interpreter that already has `saspy`/`pandas` keeps this a
one-interpreter notebook.

Nothing else to install: `saspy` and `pandas` are already in that venv,
and Java is the portable JRE at the repo root (`../jre/`), which
`config/sascfg_personal.py` resolves by itself.

In [2]:
!$PY -m pip install -q anthropic
!$PY -c "import anthropic, saspy, pandas; print('anthropic', anthropic.__version__, '| saspy', saspy.__version__, '| pandas', pandas.__version__)"


[REAL OUTPUT -- executed on this box, 2026-09-22]
anthropic 1.7.0 | saspy 5.108.7 | pandas 3.0.1


## Setup 2: Your Anthropic API key

Only for the `claude_driver.py` path -- the interactive Claude Code route
further down needs no key at all.

Best: `export ANTHROPIC_API_KEY=sk-ant-...` in the shell before starting
this kernel (or put it in your shell profile), so it never touches a
notebook cell. The SDK also reads an `ant auth login` profile from
`~/.config/anthropic/`.

The cell below falls back to a `getpass` prompt if the variable isn't set:
the value goes into this kernel's environment only, never to disk and
never into the saved notebook. Don't paste the key into a cell instead.

The check is a `models.retrieve` call -- it proves the key works and prints
the model's real limits without spending tokens on a generation. It runs
under `$PY`, which inherits this kernel's environment.

In [3]:
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (or Enter to skip): ").strip()
    if not os.environ["ANTHROPIC_API_KEY"]:
        del os.environ["ANTHROPIC_API_KEY"]
        print("No key set -- the driver cells will fail. Use the interactive route instead.")

MODEL = "claude-opus-5"
EFFORT = "high"          # output_config.effort: low | medium | high | xhigh | max
os.environ.update(MODEL=MODEL, EFFORT=EFFORT)

In [4]:
!$PY -c "import anthropic, os; m = anthropic.Anthropic().models.retrieve(os.environ['MODEL']); \
print('model:  ', m.id, '|', m.display_name); \
print('context:', getattr(m, 'max_input_tokens', '?'), 'in /', getattr(m, 'max_tokens', '?'), 'out')"


[SIMULATED OUTPUT -- needs ANTHROPIC_API_KEY; see README.md]
model:   claude-opus-5 | Claude Opus 5
context: 200000 in / 64000 out


## Setup 3: SAS OnDemand for Academics (ODA)

Already configured on this box: `~/.authinfo` holds the credentials and
`config/sascfg_personal.py`'s `iomhost` list is filled in for a
US-region/usw2 account (config1's `SETUP.md` has the Europe and Asia
Pacific host names if yours differs).

If you ever need to recreate it, do it in a terminal, not in a saved cell:

```bash
echo "oda user YOUR_ODA_EMAIL password YOUR_ODA_PASSWORD" >> ~/.authinfo
chmod 600 ~/.authinfo
```

This is the step config3 benefits from MOST -- it supplies the ground
truth that gives config3 its accuracy advantage over configs 1/2. Without
it the run still works (the `sas_column_metadata` tool reports the
failure, Claude proceeds on the static scan alone and says so in the
description), but the "tool access" story the plan wants to tell isn't
being exercised. ODA is for academic/non-commercial use -- check current
terms before pushing anything work-adjacent there.

---
## The tools, one at a time

Demonstrated on `prog900_estab.sas`, the same program config1's notebook
uses, so all three configs produce comparable output for the same program.

**You don't have to run these three cells.** `claude_driver.py` calls the
same code as tools; these are here so a surprising dictionary entry is
debuggable.

In [5]:
PROGRAM = "prog900_estab"
SAS_FILE = "../eval-programs/programs/prog900_estab.sas"
os.environ.update(PROGRAM=PROGRAM, SAS_FILE=SAS_FILE)
print(PROGRAM, "->", SAS_FILE)

[REAL OUTPUT -- executed on this box, 2026-09-22]
prog900_estab -> ../eval-programs/programs/prog900_estab.sas


### `sas_column_metadata` -- ground truth from SAS itself

Submits the program in a live ODA session, then harvests
`dictionary.columns`/`dictionary.tables` for WORK: real name, type,
length, format, informat, label and observation counts. Takes tens of
seconds (a real SAS session start), and is allowed to fail -- a program
whose hardcoded paths don't resolve outside production often still
materializes useful earlier datasets, so `run_errors` alongside usable
`columns` is a partial success worth keeping, not a failure.

Expect zero non-empty labels on this corpus: `eval-programs/` is
deliberately unlabelled, terse legacy-style SAS. Ground truth still pins
down every `type` and `length`, which is most of what configs 1/2 get
wrong by guessing.

In [6]:
!$PY sas_metadata.py $SAS_FILE --out /tmp/column_metadata.json
!$PY -c "import json; d=json.load(open('/tmp/column_metadata.json')); \
print('tables: ', sorted(d['tables'])); \
print('columns:', len(d['columns']), '| labelled:', sum(1 for c in d['columns'] if c['label'])); \
print('errors: ', d['run_errors'] or 'none'); \
print(json.dumps(d['columns'][:3], indent=1))"


[REAL OUTPUT -- executed on this box, 2026-09-22] -- live SAS OnDemand session, 18.5 s
Using SAS Config named: oda
SAS Connection established. Subprocess id is 3124172

connected: Access Method         = IOM
SAS Config name       = oda
SAS Config file       = /internal/sas-doc-gen-project/config3-frontier-skills/config/sascfg_personal.py
WORK Path             = /saswork/SAS_work96F70001CB65_odaws02-usw2.oda.sas.com/SAS_workF1640001CB65_odaws02-usw2.oda.sas.com/
SAS Version           = 9.04.01M8P02222023
SASPy Version         = 5.108.7
Teach me SAS          = False
Batch                 = False
Results               = Pandas
SAS Session Encoding  = utf-8
Python Encoding value = utf-8
SAS process Pid value = 117605
SASsession started    = Tue Sep 22 10:16:29 2026


SAS Connection terminated. Subprocess id was 3124172
wrote /tmp/column_metadata.json (39 column rows across 6 table(s))
tables:  ['AGG1', 'D1', 'D2', 'HOLD', 'J1', 'O1']
columns: 39 | labelled: 0
errors:  none
[
 {
  "program_

### `header_comments` and `static_identifier_scan`

Human-authored context (`header_found: false` is the normal result on this
corpus -- it has zero header comments on purpose), then the regex
allow-list of every identifier that demonstrably appears in the source.

In [7]:
!python3 header_extract.py $SAS_FILE
!python3 extract.py $SAS_FILE

[REAL OUTPUT -- executed on this box, 2026-09-22]
{
  "header_found": false,
  "fields": {},
  "raw_header_text": "",
  "inline_glosses": {}
}
datasets (6): AGG1, D1, D2, HOLD, J1, O1
params (3): lb, p, thr
variables (16): AGG1, DROP, EMPL, ESTID, HIR, JO, OUT, PER, SELECT, SEP, SEPR, ST, SUM, WGTF, _FREQ_, _TYPE_


---
## Document one program

`claude_driver.py` runs the loop: Claude gets the source plus the four
tools, calls what it needs, and returns the dictionary JSON in
`schema.py`'s shape. The driver then hands that JSON to the same two
scripts the interactive path uses --

- `write_dictionary.py` (`--catalog`): cross-checks every identifier
  against ground truth + the static scan, stamps misses as
  `guardrail_flagged`, and upserts into the local JSONL catalog;
- `save_prediction.py` (`--preds-out`): writes the
  `.pred.json`/`.meta.json` pair `results/score.py` reads, with the
  **measured** `elapsed_sec` and the **real** `cost_usd`.

-- so there's one implementation of validation and cataloguing, and no
`--elapsed-sec 90.0` placeholder to regret later.

If it reports guardrail flags, don't just push: check whether Claude
invented a name or `extract.py`'s regex missed a real one (it does miss
some forms), and fix the JSON before continuing.

Written per program: `claude-runs/<program>.dictionary.json` (the
dictionary), `claude-runs/<program>.run.json` (turns, repair turns,
tool-call counts, token usage, cost), `claude-runs/metadata/<program>.json`
(the harvested ground truth, already in the layout `run_eval.py
--extra-source-dir` reads).

In [8]:
PREDS = "../results/preds/config3-frontier-skills"
os.environ["PREDS"] = PREDS

!$PY claude_driver.py $SAS_FILE \
    --model $MODEL --effort $EFFORT \
    --out claude-runs --catalog catalog/ --preds-out $PREDS

[SIMULATED OUTPUT -- needs ANTHROPIC_API_KEY; see README.md]

=== [1/1] prog900_estab ===
catalog: catalog/ -> program_summary=1, macro_params=3, data_dictionary=12, column_metadata=39 (prog900_estab)
no guardrail flags.
wrote prog900_estab.pred.json and .meta.json to ../results/preds/config3-frontier-skills
prog900_estab: 67.6s, $0.1200, 3 api turn(s), tools={'sas_column_metadata': 1, 'header_comments': 1, 'static_identifier_scan': 1}, ground truth=True

1/1 documented
measured cost: $0.1200 total, $0.1200 mean per program (1 priced)


In [9]:
import json

run = json.load(open("claude-runs/%s.run.json" % PROGRAM))
print(json.dumps(run, indent=2))
print("\n--- authored dictionary (first 60 lines) ---")
print("\n".join(open("claude-runs/%s.dictionary.json" % PROGRAM).read().splitlines()[:60]))

[SIMULATED OUTPUT -- needs ANTHROPIC_API_KEY; see README.md]
{
  "program_name": "prog900_estab",
  "requested_model": "claude-opus-5",
  "effort": "high",
  "structured_output": false,
  "sas_tool_offered": true,
  "ground_truth_used": true,
  "elapsed_sec": 67.64,
  "cost_usd": 0.120035,
  "api_turns": 3,
  "repair_turns": 0,
  "tool_calls": {
    "sas_column_metadata": 1,
    "header_comments": 1,
    "static_identifier_scan": 1
  },
  "usage": {
    "input_tokens": 10367,
    "output_tokens": 2728,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "models_served": [
      "claude-opus-5"
    ],
    "unpriced_models": []
  }
}

--- authored dictionary (first 60 lines) ---
{
  "program_name": "prog900_estab.sas",
  "program_summary": {
    "description": "Builds an analysis-ready establishment file for a single reference period by merging the control/weight file onto collected microdata, keeping establishments above a minimum employment threshold, and comput

### Doing this step interactively instead

The driver isn't the only way, and on a single program it isn't
necessarily the best one. In a Claude Code session in this repo, just ask:

> document `eval-programs/programs/prog900_estab.sas` using the
> sas-data-dictionary skill

The skill triggers, runs these same scripts, and writes the same JSON --
then persist it exactly as the driver does:

```bash
python3 write_dictionary.py --program-name prog900_estab \
    --source ../eval-programs/programs/prog900_estab.sas \
    --dictionary /tmp/dictionary.json \
    --column-metadata /tmp/column_metadata.json --catalog catalog/

python3 save_prediction.py --program-name prog900_estab \
    --dictionary /tmp/dictionary.json --elapsed-sec <measured> \
    --used-proc-contents --out ../results/preds/config3-frontier-skills
```

What you lose: the measured cost (an interactive session has no metered
per-call cost, so `cost_usd` is honestly `not recorded` -- **not** `$0`)
and unattended runs. What you gain: you can interrogate a surprising entry
as it's written. Same config either way -- say which one produced the
numbers you report.

---
## Document all 20

One command, one program at a time. A per-program failure (API error, a
dropped SAS session, output that still misses the schema after its repair
attempts) is logged and skipped rather than aborting the run -- same
policy as config1's batch mode -- and the last line lists what failed so
you can re-run just those.

Each program opens its own ODA session for the harvest, so budget on the
order of a minute per program, most of it SAS rather than Claude.

Add `--limit N` for a smoke test, but a partial run is **not** a result:
`score.py` scores against all 20 gold files and counts every missing
prediction as a program that produced no output.

The driver prints measured cost per program and the total. Check the
single-program number above before launching this.

In [10]:
!$PY claude_driver.py --dir ../eval-programs/programs \
    --model $MODEL --effort $EFFORT \
    --out claude-runs --catalog catalog/ --preds-out $PREDS

[SIMULATED OUTPUT -- needs ANTHROPIC_API_KEY; see README.md]

=== [1/20] prog900_estab ===
catalog: catalog/ -> program_summary=1, macro_params=3, data_dictionary=12, column_metadata=39 (prog900_estab)
no guardrail flags.
wrote prog900_estab.pred.json and .meta.json to ../results/preds/config3-frontier-skills
prog900_estab: 67.6s, $0.1200, 3 api turn(s), tools={'sas_column_metadata': 1, 'header_comments': 1, 'static_identifier_scan': 1}, ground truth=True

=== [2/20] prog901_hhold ===
catalog: catalog/ -> program_summary=1, macro_params=2, data_dictionary=15, column_metadata=52 (prog901_hhold)
no guardrail flags.
wrote prog901_hhold.pred.json and .meta.json to ../results/preds/config3-frontier-skills
prog901_hhold: 53.6s, $0.1365, 3 api turn(s), tools={'sas_column_metadata': 1, 'header_comments': 1, 'static_identifier_scan': 1}, ground truth=True

=== [3/20] prog902_estcost ===
catalog: catalog/ -> program_summary=1, macro_params=4, data_dictionary=13, column_metadata=24 (prog902_estco

### The second run the plan asks for: no ground truth

Same model, same prompt, `--no-sas-tool` -- the `sas_column_metadata`
tool is simply not offered, so Claude works from the source text and the
static scan alone, which is what configs 1 and 2 have. That pair of rows
(with tool access / without) is the fairer isolate of model quality,
since config3's ground truth is something the other two configs
structurally cannot get.

It also skips every SAS session, so it's much faster than the run above.
Writes to a **different** preds directory, so the two runs can never be
pooled into one results row by accident.

In [11]:
PREDS_NOGT = "../results/preds/config3-frontier-skills-nogt"
os.environ["PREDS_NOGT"] = PREDS_NOGT

!$PY claude_driver.py --dir ../eval-programs/programs --no-sas-tool \
    --model $MODEL --effort $EFFORT \
    --out claude-runs-nogt --preds-out $PREDS_NOGT

[SIMULATED OUTPUT -- needs ANTHROPIC_API_KEY; see README.md]

=== [1/20] prog900_estab ===
wrote prog900_estab.pred.json and .meta.json to ../results/preds/config3-frontier-skills-nogt
prog900_estab: 37.4s, $0.0819, 2 api turn(s), tools={'header_comments': 1, 'static_identifier_scan': 1}, ground truth=False

=== [2/20] prog901_hhold ===
wrote prog901_hhold.pred.json and .meta.json to ../results/preds/config3-frontier-skills-nogt
prog901_hhold: 53.3s, $0.1163, 3 api turn(s), tools={'header_comments': 1, 'static_identifier_scan': 1}, ground truth=False

=== [3/20] prog902_estcost ===
wrote prog902_estcost.pred.json and .meta.json to ../results/preds/config3-frontier-skills-nogt
prog902_estcost: 50.6s, $0.0847, 2 api turn(s), tools={'header_comments': 1, 'static_identifier_scan': 1}, ground truth=False

=== [4/20] prog903_estcost ===
wrote prog903_estcost.pred.json and .meta.json to ../results/preds/config3-frontier-skills-nogt
prog903_estcost: 48.7s, $0.0893, 2 api turn(s), tools={'heade

---
## Score the run(s)

`run_eval.py` stamps a `.provenance.json` sidecar recording exactly which
eval corpus was scored, and `--table` marks a row **STALE** rather than
printing numbers that no longer refer to the corpus on disk. Config3's own
2026-09-17 run is why that check exists: it read `1.00` on every metric,
while the same predictions scored against the current gold read `0.79`
variable F1 and `0.00` macro F1. See
`../results/outputs/stale-2026-09-17/README.md`.

`--extra-source-dir` points at the ground truth the driver saved, per
`PLAN.md` section 3's "for config 3, PROC CONTENTS output also counts as a
valid source" -- a name that appears in SAS's metadata but not in the
source text is then not a hallucination. The provenance sidecar records
that it was used, so a row scored WITH ground truth is never silently
compared against one scored without it.

Unlike the Colab configs, the judge *can* run here: it's a local Ollama
model (`../results/llm_judge.py`, default `gemma3:1b`), which is the
config1 runtime on this box -- a different model family from config3, as
the plan requires to avoid self-grading bias. Add `--run-judge` once
`/internal/e2b-gemma`'s server is up; without it the table prints
`not run` for Description score rather than a made-up number.

In [12]:
!python3 ../results/run_eval.py --config config3-frontier-skills \
    --pred-dir $PREDS --extra-source-dir claude-runs/metadata \
    --out-prefix ../results/outputs/config3-frontier-skills

[REAL OUTPUT -- executed on this box, 2026-09-22] -- the real scorer, over SIMULATED predictions and the REAL SAS harvest
[config3-frontier-skills] wrote config3-frontier-skills.scores.jsonl (20 programs)
[config3-frontier-skills] wrote config3-frontier-skills.provenance.json (corpus gold_digest=f27fb0130faf)
[config3-frontier-skills] hallucination check also allowed 20/20 programs' real SAS metadata from metadata as a valid source


In [13]:
# The no-ground-truth run: no --extra-source-dir, because there wasn't any.
!test -d $PREDS_NOGT && python3 ../results/run_eval.py \
    --config config3-frontier-skills-nogt --pred-dir $PREDS_NOGT \
    --out-prefix ../results/outputs/config3-frontier-skills-nogt \
    || echo "skipped -- run the --no-sas-tool cell first"


[REAL OUTPUT -- executed on this box, 2026-09-22] -- the real scorer, over SIMULATED predictions
[config3-frontier-skills-nogt] wrote config3-frontier-skills-nogt.scores.jsonl (20 programs)
[config3-frontier-skills-nogt] wrote config3-frontier-skills-nogt.provenance.json (corpus gold_digest=f27fb0130faf)


In [14]:
!python3 ../results/run_eval.py --table \
    --scores ../results/outputs/config3-frontier-skills.scores.jsonl \
    --meta-dir $PREDS \
    --label "Config 3: Claude + SAS tool access ($MODEL, effort=$EFFORT)" 

[REAL OUTPUT -- executed on this box, 2026-09-22] -- the real table builder, over SIMULATED predictions
eval corpus on disk: 20 gold / 20 programs (gold_digest f27fb0130faf)

| Config | Schema validity | Variable F1 | Macro F1 | I/O F1 | Hallucination rate | Description score | Time/program | Cost/program |
|---|---|---|---|---|---|---|---|---|
| Config 3: Claude + SAS tool access (claude-opus-5, effort=high) | 1.00 [1.00, 1.00] | 0.99 [0.98, 1.00] | 1.00 [1.00, 1.00] | 1.00 [1.00, 1.00] | 0.00 [0.00, 0.00] | not run | 61.7s | $0.1281 |

Config 3: Claude + SAS tool access (claude-opus-5, effort=high): 20 program(s), [1] run(s) each; 20/20 scored outputs parsed as JSON, 20/20 passed schema.validate().
    NOTE: 1 run only. PLAN.md asks for 3 runs per config to separate model variance from program difficulty -- the CI below reflects program difficulty alone.

Hallucination rate convention: a program that produced no parseable output is scored 1.00 (worst case), not skipped -- see score.p

## Push the catalog to SAS as real datasets

Writes/replaces `PROGRAM_SUMMARY`, `MACRO_PARAMS`, `DATA_DICTIONARY` and
`COLUMN_METADATA` in the target SAS library (default `SASUSER` on ODA --
pass `--libname`/`--libpath` for a different, permanent, custom-path
library). Each run mirrors the CURRENT full contents of the local catalog,
so it's safe to re-run after documenting more programs; it doesn't append
duplicates.

Check what's in `catalog/` first. The catalog built before the 2026-09-21
rewrite of `../eval-programs/` was moved aside to
`catalog-stale-2026-09-17/`; pushing that one would put documentation for
programs that no longer exist into `SASUSER` under current program names.

Optional -- skip it if you only want the local JSON/catalog output.

In [15]:
!ls catalog/ 2>/dev/null || echo "no catalog/ yet -- run the driver with --catalog first"
!$PY push_to_oda.py --catalog catalog/

[SIMULATED OUTPUT -- needs ANTHROPIC_API_KEY; see README.md] -- NOT run: this writes to your live SASUSER library
column_metadata.jsonl  data_dictionary.jsonl  macro_params.jsonl  program_summary.jsonl
program_summary: 20 row(s) in local catalog
macro_params: 60 row(s) in local catalog
data_dictionary: 254 row(s) in local catalog
column_metadata: 741 row(s) in local catalog
Using SAS Config named: oda
SAS Connection established. Subprocess id is 3142118
wrote SASUSER.PROGRAM_SUMMARY (20 rows, 9 cols)
wrote SASUSER.MACRO_PARAMS (60 rows, 8 cols)
wrote SASUSER.DATA_DICTIONARY (254 rows, 8 cols)
wrote SASUSER.COLUMN_METADATA (741 rows, 10 cols)
done
SAS Connection terminated. Subprocess id was 3142118


---
## What the driver does and doesn't constrain

- **No decoding constraint by default.** Configs 1 and 2 get the schema as
  prompt *text*; so does config3. `--structured-output` will constrain
  decoding to the JSON schema, which makes schema validity trivially 1.00
  -- fine in a production pipeline, but it would make that column measure
  the harness instead of the model, so it's off by default and worth
  declaring if you turn it on.
- **Repair turns are counted, not hidden.** If the returned JSON misses
  `schema.py`'s shape, the driver re-asks with the validation errors (up
  to `--max-repairs`, default 2) and records `repair_turns` in
  `<program>.run.json`. A config that needed repairs is not the same as
  one that didn't.
- **Cost is priced per turn, at the model that actually served it.**
  `--no-fallbacks` disables the server-side refusal fallback if you'd
  rather a policy decline fail loudly than be answered by another model;
  either way `usage.models_served` records who answered, and an
  unrecognized model id makes `cost_usd` null rather than wrong. The
  `PRICES` table in `claude_driver.py` was checked 2026-09-22 -- re-check
  before quoting the cost column.
- **`--effort`** (`output_config.effort`) is the quality/spend dial:
  `high` is the default and the sweet spot here; `max` costs more for
  marginal gain, `low`/`medium` are the cheaper step-downs to measure
  before assuming you need more.
- **The harvest no longer reports its own scratch tables.**
  `sas_metadata.py` builds `work._sdgcols`/`_sdgtabs` to hold the
  dictionary query's output, and `dictionary.tables` saw them while they
  were being created -- so the ground truth used to list `_SDGCOLS` as one
  of the program's datasets. It's now excluded unconditionally, separately
  from `--exclude-pattern` (which still defaults to excluding nothing,
  because legacy SAS really does name real datasets with a leading
  underscore).

## The asymmetry to report honestly

Config 3's `dictionary.columns` access is real ground truth the other two
configs structurally cannot get: config1 has no SAS connection at all by
design, and config2's fine-tuning happens offline with no live SAS session
at inference time either. If config3 scores much higher on
`type_length_accuracy` or `hallucination_rate`, that may be measuring "has
ground truth" more than "is a better model" -- which is exactly why the
`--no-sas-tool` run exists. Report the pair, and say plainly that
config3's axis is tool access, not model quality alone.

## If you do have to run this in Colab

Nothing here is Colab-hostile, but you'd be rebuilding the environment
this box already has, and config3 gains no free compute from it. If you
must:

1. clone the whole repo (`git clone --depth 1 ...`, then `cd` into this
   folder) -- `../eval-programs/` and `../results/` have to be siblings;
2. `pip install -r requirements.txt` into the kernel and use
   `sys.executable` as `$PY`;
3. `apt-get install default-jdk` -- `../jre/` is 136 MB and gitignored, so
   a fresh clone doesn't have it and SASPy's IOM connection needs Java;
4. put `ANTHROPIC_API_KEY`, `ODA_USER` and `ODA_PASS` in the Secrets panel
   and write `~/.authinfo` from the latter two at the start of the session;
5. zip `claude-runs/`, `catalog/`, `../results/preds/` and
   `../results/outputs/` and download them before the runtime recycles --
   those files ARE the run, and re-creating them costs the API spend again.

`config1-gemma-cpu/` and `config2-qlora-gpu/` are the notebooks that
genuinely want Colab: they need a free CPU runtime and a free T4
respectively.

## Don't confuse this with config1

`config1-gemma-cpu/document_sas.py` is a separate, independent pipeline
(small Gemma model, no ground-truth SAS metadata at all, its own regex
guardrail) used to establish the air-gapped floor for comparison. Don't
mix the two catalogs by hand-editing; they write to different `catalog/`
directories under their own config folders by design, specifically so
config1 and config3 runs never clobber each other.